# 02e Estimate: a second horizon, the total of the next 12 months

**Purpose:** SEM and the cantons also plan places and budgets for a whole year. We forecast the **total number of applications in the next 12 months** and check whether the extra data blocks help at this longer horizon, where conflicts and routes might matter more than at two months.
**Input:** `../data/processed/asylum_clean.csv` and the three files of `01b_preprocess_indicators.ipynb` (Part B).
**Output:** `../data/processed/backtest_forecasts_one_year.csv`. The errors are evaluated in `03_evaluate.ipynb`, Part B (section 18).

This is a secondary analysis on the backtest, decided after the test. We fixed the design below before running it.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge

np.random.seed(42)
plt.style.use("seaborn-v0_8-muted")

## 1. Target, features and timing

- **Target** at origin $t$, with $S_{a..b}$ the sum of the monthly applications from month $a$ to month $b$: the change of the log annual total, $\log S_{t+1..t+12} - \log S_{t-11..t}$. The forecast of the total is $S_{t-11..t} \cdot e^{\hat\Delta}$.
- **Simple rules:** (1) "next 12 months = last 12 months", which is exactly our 12-month average times 12 (`02_estimate.ipynb`); (2) the long-run average, 12 times the average of all known months.
- **Own-history features:** the growth of the annual total, $\log S_{t-11..t} - \log S_{t-23..t-12}$, and the growth of the last three months against the same three months a year earlier.
- **Extra blocks** (one at a time, same timing as in `02c`): the conflict index over 6 and 12 months (deaths up to $t-1$); the growth of the 12-month total of the Mediterranean routes (up to $t$); the growth of the 12-month total of the neighbours (up to $t-1$).
- **Models:** linear regression and Ridge (`alpha` chosen as in `02b`, but with a gap of 12 months between training and validation rows, because the targets now cover 12 months).

The targets of neighbouring origins overlap by up to 11 months, so 94 origins contain only about 8 independent years. Differences between models will be hard to separate from noise.

In [2]:
clean = pd.read_csv("../data/processed/asylum_clean.csv")
clean["month"] = pd.to_datetime(clean["month"])
clean = clean.set_index("month")
months = clean.index
y = clean["applicants_total"].to_numpy(dtype=float)
T = len(y)
H = 12

def load(path):
    data = pd.read_csv(path)
    data["month"] = pd.to_datetime(data["month"])
    return data.set_index("month")

extra = load("../data/processed/extra_indicators_clean.csv").reindex(months)
shares = load("../data/processed/nationality_shares_clean.csv").reindex(months)
intensity = load("../data/processed/conflict_intensity_clean.csv")
neighbours = np.exp(extra["log_neighbours"].to_numpy(dtype=float))
med = np.expm1(extra["log_med_routes"].to_numpy(dtype=float))

def window_sum(x, end, n):
    # sum of x over the months end-n+1 ... end (NaN if outside the data)
    return np.nan if end - n + 1 < 0 or end >= len(x) else x[end - n + 1:end + 1].sum()

def growth_12(x, end):
    return np.log(window_sum(x, end, 12)) - np.log(window_sum(x, end - 12, 12)) if end >= 23 else np.nan

S_past = np.array([window_sum(y, t, 12) for t in range(T)])
S_next = np.array([window_sum(y, t + 12, 12) for t in range(T)])
target = np.log(S_next) - np.log(S_past)

def conflict_change(t, k):
    w = shares.iloc[t]
    if w.isna().all():
        return np.nan
    top = w.nlargest(10)
    m1, m0 = months[t] - pd.DateOffset(months=1), months[t] - pd.DateOffset(months=1 + k)
    weighted = top * (intensity.loc[m1, top.index] - intensity.loc[m0, top.index])
    return float(weighted.sum()) if weighted.notna().all() else np.nan    # NaN if a month is not in UCDP yet

own = {"growth_12_months": np.array([growth_12(y, t) for t in range(T)]),
       "growth_3_months_yoy": np.array([np.log(window_sum(y, t, 3)) - np.log(window_sum(y, t - 12, 3)) if t >= 14 else np.nan
                                        for t in range(T)])}
blocks = {
    "own history only": {},
    "conflict": {"conflict_6_months": np.array([conflict_change(t, 6) for t in range(T)]),
                 "conflict_12_months": np.array([conflict_change(t, 12) for t in range(T)])},
    "Med routes": {"med_growth_12_months": np.array([growth_12(med, t) for t in range(T)])},
    "neighbours": {"neighbours_growth_12_months": np.array([growth_12(neighbours, t - 1) for t in range(T)])},
}

FIRST_ROW = int(np.where(months == pd.Timestamp("2010-12-01"))[0][0])     # every feature exists from here
FIRST_ORIGIN = int(np.where(months == pd.Timestamp("2015-12-01"))[0][0])
LAST_ORIGIN = int(np.where(months == pd.Timestamp("2023-09-01"))[0][0])    # its 12 months end in 2024-09
for name, ex in blocks.items():
    for k, v in {**own, **ex}.items():
        assert not np.isnan(v[FIRST_ROW:LAST_ORIGIN + 1]).any(), (name, k)
print("Training rows from", months[FIRST_ROW].strftime("%Y-%m"), "| origins", months[FIRST_ORIGIN].strftime("%Y-%m"),
      "to", months[LAST_ORIGIN].strftime("%Y-%m"), "|", LAST_ORIGIN - FIRST_ORIGIN + 1, "origins")

Training rows from 2010-12 | origins 2015-12 to 2023-09 | 94 origins


The first origin is December 2015: the training rows start in December 2010 (when every feature exists), a row can be used only when its 12-month target is known, and Ridge needs two validation blocks of 12 months with a gap of 12 months. The last origin is September 2023, whose 12 months end in September 2024, before the test period. All models use the same training rows.

## 2. The backtest

In [3]:
ALPHAS = [0.1, 1, 10, 100, 1000, 10000]

def design(ex, rows):
    cols = {**own, **ex}
    return np.column_stack([cols[k][rows] for k in cols])

def choose_alpha(ex, t):
    last_row = t - H                                     # last row whose 12-month target is known at t
    scores = []
    for a in ALPHAS:
        mse = []
        for bl in range(2):
            start = last_row - 24 + 1 + bl * 12
            valid = np.arange(start, start + 12)
            train = np.arange(FIRST_ROW, start - H + 1)
            m = make_pipeline(StandardScaler(), Ridge(alpha=a)).fit(design(ex, train), target[train])
            mse.append(np.mean((target[valid] - m.predict(design(ex, valid))) ** 2))
        scores.append(np.mean(mse))
    return ALPHAS[int(np.argmin(scores))]

rows = []
for t in range(FIRST_ORIGIN, LAST_ORIGIN + 1):
    train = np.arange(FIRST_ROW, t - H + 1)
    assert train.max() + H <= t                          # no training target after the origin
    rows.append({"model": "Next 12 months = last 12 months", "origin": months[t], "forecast": S_past[t], "actual": S_next[t]})
    rows.append({"model": "Long-run average", "origin": months[t], "forecast": 12 * y[:t + 1].mean(), "actual": S_next[t]})
    for name, ex in blocks.items():
        X_train, X_t = design(ex, train), design(ex, np.array([t]))
        lr = make_pipeline(StandardScaler(), LinearRegression()).fit(X_train, target[train])
        rd = make_pipeline(StandardScaler(), Ridge(alpha=choose_alpha(ex, t))).fit(X_train, target[train])
        suffix = "" if name == "own history only" else f" + {name}"
        for label, m in [("Linear regression", lr), ("Ridge", rd)]:
            rows.append({"model": label + suffix, "origin": months[t],
                         "forecast": S_past[t] * np.exp(m.predict(X_t)[0]), "actual": S_next[t]})

fc = pd.DataFrame(rows)
fc["abs_error"] = (fc["actual"] - fc["forecast"]).abs()
fc["abs_pct_error"] = 100 * fc["abs_error"] / fc["actual"]
print("Forecasts per model:", fc.groupby("model").size().unique())

Forecasts per model: [94]


## 3. Save the forecasts

In [4]:
fc.to_csv("../data/processed/backtest_forecasts_one_year.csv", index=False)
print("Saved", len(fc), "forecasts")

Saved 940 forecasts


## Summary

- **Horizon:** the total of the next 12 months, for annual planning; 94 origins (2015-12 to 2023-09), all models on the same training rows, no training target after the origin (`assert`).
- **Models:** two simple rules ("next 12 months = last 12 months", the long-run average), the linear regression and Ridge on the series' own past, and each regression with one extra block (conflict index, Mediterranean routes, neighbours).
- **Output:** 940 forecasts in `../data/processed/backtest_forecasts_one_year.csv`, evaluated in `03_evaluate.ipynb`, Part B (section 18).
- **Caveat:** the 12-month targets overlap, so the 94 origins hold only about 8 independent years.